# Indic Speech Deepfake Detection

Detecting **AI-generated (TTS) speech** versus real speech across **16 Indian languages** by fine-tuning **DistilHuBERT**, a compact self-supervised speech model.

- **Data:** IndicTTS Deepfake Challenge dataset (Hugging Face `SherryT997/IndicTTS-Deepfake-Challenge-Data`): 33,737 clips in 16 languages (31,102 labelled training clips, 2,635 unlabelled test clips), about 63 hours of audio
- **Metric:** ROC-AUC on the predicted probability that a clip is synthetic
- **Result:** ROC-AUC **0.99999** on the held-out test set; ranked **7th of 201** teams in the IIT Madras course challenge

**Pipeline:** language-balanced sampling → 1.5-second clips with waveform normalisation → DistilHuBERT with a classification head (feature encoder frozen) → 5 epochs of fine-tuning → probability scores for the test set.

## 1. Setup

In [ ]:
# Install required libraries
!pip install datasets transformers torch evaluate scipy pandas numpy tqdm

from dataclasses import dataclass
from typing import Dict, List, Union

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from evaluate import load
from scipy.special import softmax
from tqdm import tqdm
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, TrainingArguments, Trainer

## 2. Load the dataset

In [ ]:
# Official dataset for the challenge
dataset = load_dataset("SherryT997/IndicTTS-Deepfake-Challenge-Data")

train_data = dataset["train"]  # contains 'is_tts' labels (1 = synthetic, 0 = real)
test_data = dataset["test"]    # 'is_tts' is -1 for all rows

## 3. Load DistilHuBERT
DistilHuBERT is a distilled version of HuBERT (about 24M parameters), pre-trained on raw speech. A two-class classification head is added on top. The feature extractor normalises each waveform (zero mean, unit variance).

In [ ]:
model_id = "ntu-spml/distilhubert"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id, do_normalize=True, return_attention_mask=True)
model = AutoModelForAudioClassification.from_pretrained(model_id, num_labels=2)  # real vs synthetic

## 4. Language-balanced sampling
To fit the GPU time limit, 45% of the clips are taken **from each language separately**, so every language stays represented in the same proportion as in the full dataset.

In [5]:
unique_languages = set(train_data["language"])
fraction = 0.45
sampled_indices = []

for lang in unique_languages:
    lang_indices = [i for i, lang_val in enumerate(train_data["language"]) if lang_val == lang]
    sample_size = max(1, int(len(lang_indices) * fraction))  # at least one clip per language
    sampled_indices.extend(lang_indices[:sample_size])       # first 45% of this language's clips

sampled_dataset = train_data.select(sampled_indices)
print(sampled_dataset)

Dataset({
    features: ['text', 'id', 'language', 'is_tts', 'audio'],
    num_rows: 13993
})


## 5. Pre-process the audio
Every clip is trimmed or zero-padded to **1.5 seconds (24,000 samples at 16 kHz)**, so all inputs have the same length, then passed through the feature extractor. A 10% split is held out for validation.

In [6]:
def prepare_sample(batch):
    audio = batch["audio"]["array"]
    expected_length = 24000  # 1.5 s at 16 kHz

    # Trim or pad to 1.5 s
    if len(audio) < expected_length:
        audio = np.pad(audio, (0, expected_length - len(audio)), mode="constant")
    else:
        audio = audio[:expected_length]

    inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt", padding=True)
    batch["input_values"] = inputs.input_values[0]
    batch["labels"] = torch.tensor(batch["is_tts"], dtype=torch.float)
    return batch

In [7]:
sampled_dataset = sampled_dataset.map(prepare_sample, remove_columns=["audio", "text", "id", "language", "is_tts"])

# 90% training, 10% validation
dataset1 = sampled_dataset.train_test_split(test_size=0.1, shuffle=True, seed=42)

## 6. Batching and evaluation metrics

In [8]:
@dataclass
class DataCollatorWithPadding:
    """Pads the waveforms in a batch and converts labels to a tensor."""
    processor: AutoFeatureExtractor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        batch["labels"] = torch.tensor([feature["labels"] for feature in features], dtype=torch.long)
        return batch


data_collator = DataCollatorWithPadding(feature_extractor, padding=True)

In [9]:
accuracy_metric = load("accuracy")
precision_metric = load("precision")
recall_metric = load("recall")
f1_metric = load("f1")
roc_auc_metric = load("roc_auc")

def compute_metrics(pred):
    # Probability of class 1 (synthetic speech)
    pred_probs = softmax(pred.predictions, axis=-1)[:, 1]
    labels = pred.label_ids
    pred_labels = pred_probs.round()

    return {
        "accuracy": accuracy_metric.compute(predictions=pred_labels, references=labels)["accuracy"],
        "precision": precision_metric.compute(predictions=pred_labels, references=labels, average="binary")["precision"],
        "recall": recall_metric.compute(predictions=pred_labels, references=labels, average="binary")["recall"],
        "f1": f1_metric.compute(predictions=pred_labels, references=labels, average="binary")["f1"],
        "roc_auc": roc_auc_metric.compute(prediction_scores=pred_probs, references=labels)["roc_auc"],  # competition metric
    }

## 7. Fine-tune
The convolutional feature encoder is frozen, so only the transformer layers and the classification head are trained. Training runs for 5 epochs with mixed precision (fp16) and gradient checkpointing, evaluating on the validation split every 500 steps.

In [11]:
model.to("cuda")
model.freeze_feature_encoder()

In [12]:
training_args = TrainingArguments(
    output_dir="AsrTaskModel",
    group_by_length=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="steps",
    num_train_epochs=5,
    fp16=True,
    gradient_checkpointing=True,
    save_steps=1000,
    eval_steps=500,
    logging_steps=500,
    learning_rate=1e-4,
    weight_decay=0.005,
    warmup_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    save_strategy="steps",
    report_to="none",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset1["train"],
    eval_dataset=dataset1["test"],
    tokenizer=feature_extractor,
)

In [13]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
500,0.540600,0.330690,0.850714,0.962162,0.739612,0.836335,0.961797
1000,0.297100,0.227507,0.927143,0.968278,0.887812,0.926301,0.982598
1500,0.272900,0.138014,0.955714,0.962185,0.951524,0.956825,0.991566
2000,0.178400,0.224722,0.927143,0.917790,0.943213,0.930328,0.984941
2500,0.143200,0.123908,0.962143,0.948993,0.979224,0.963872,0.995173
3000,0.139400,0.127196,0.965000,0.966713,0.965374,0.966043,0.996347
3500,0.103900,0.151689,0.962857,0.981322,0.945983,0.963329,0.996341
4000,0.072100,0.114740,0.969286,0.949669,0.993075,0.970887,0.998408
4500,0.062300,0.103073,0.972857,0.965940,0.981994,0.973901,0.997802
5000,0.032000,0.074509,0.980714,0.984658,0.977839,0.981237,0.998845


Could not locate the best model at AsrTaskModel/checkpoint-6500/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


TrainOutput(global_step=7875, training_loss=0.12154624806510078, metrics={'train_runtime': 1846.6302, 'train_samples_per_second': 34.097, 'train_steps_per_second': 4.265, 'total_flos': 2.1479035260816e+17, 'train_loss': 0.12154624806510078, 'epoch': 5.0})

Validation ROC-AUC rose from 0.962 at step 500 to **0.999** by the end of training, with F1 around 0.99. The warning above means the best checkpoint was not reloaded, so the model used for prediction is the one from the final step.

## 8. Predict on the test set
Each test clip is passed through the model at its **full length**, and the softmax probability of the synthetic class is saved as `is_tts`.

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

submission_results = []
for sample in tqdm(test_data):
    inputs = feature_extractor(sample["audio"]["array"], sampling_rate=16000, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probabilities = softmax(logits.cpu().numpy(), axis=-1)
    is_tts_prob = round(probabilities[0, 1], 3)  # probability that the clip is synthetic
    submission_results.append([sample["id"], is_tts_prob])

submission_df = pd.DataFrame(submission_results, columns=["id", "is_tts"])
submission_df.to_csv("submission.csv", index=False)
print("Submission file saved.")

100%|██████████| 2635/2635 [00:59<00:00, 44.25it/s]


Submission file saved.


## 9. Combining runs
For the final competition submission, the notebook was run with **different training fractions** (the run saved here uses 45%), and the resulting submission files were combined with a **weighted average** of the predicted probabilities. The function below shows that step; the file names and weights are placeholders.

In [ ]:
def blend_submissions(files, weights):
    """Weighted average of the is_tts probabilities from several submission files."""
    frames = [pd.read_csv(f).set_index("id") for f in files]
    blended = sum(w * df["is_tts"] for df, w in zip(frames, weights)) / sum(weights)
    return blended.round(3).reset_index()

# Example: blend_submissions(["run_a.csv", "run_b.csv"], weights=[0.6, 0.4]).to_csv("submission.csv", index=False)